# 파이썬으로 MLP 구현하기

## 구현 개요
- 자유로운 신경망 구성을 위해 `class MLP` 를 정의
- 신경망의 학습에 필요한 순전파, 역전파, 오차함수 등을 `class MLP`의 메소드들로 구현
- 행렬 곱 연산을 위해서는 배열이 2차원이어야 해서, `reshape(-1, 1)`로 벡터를 열벡터 형태로 손쉽게 변환
- C언어로 구현했다면 같은 문제(레이어의 노드들의 값을 저장하는 배열, 델타 값 등)를 이중 포인터가 아닌 삼중 포인터로 선언해 해결해야 했을 부분을, 파이썬에서는 리스트와 배열로 간단히 처리
- 전치행렬 변환, 행렬의 사칙연산 등은 numpy에 이미 구현되어 있어 별도 구현 없이 편리하게 계산


In [7]:
import io, sys

test_input = """4
3 3 2 2
0.1
0.1
0.5 0.3 0.8
0.1 0.2 0.3
0.4 0.5 0.6
0.7 0.8 0.9
0.2 0.3 0.4
0.5 0.6 0.7
0.3 0.4
0.5 0.6
1.0 0.0
2
"""

sys.stdin = io.StringIO(test_input)

## 코드 설명

### 활성화 함수
- `sigmoid(x)`

### 출력 포맷 헬퍼
- `fmt_vec`, `fmt_mat`: numpy 벡터/행렬을 소수점 4자리까지 보기 좋게 문자열로 변환하는 함수 (계산 로직과는 무관, 출력용)

### MLP 클래스
- `__init__`: `nodes` 리스트에 맞춰 각 층을 0으로 초기화하고, 편향과 학습률을 저장
- `setting_input_layer`: 입력값을 열벡터(`reshape(-1, 1)`)로 변환해 첫 번째 층에 저장
- `setting_weights`: 미리 구성한 가중치 행렬 리스트를 객체에 저장
- `forward`: 각 층마다 `가중치 @ 이전 층` 연산 후 편향을 더하고, 출력층 직전까지는 시그모이드를 적용 (출력층은 항등함수). 계산이 끝나면 층별 결과를 출력
- `error`: 목표값과 출력층 값의 오차제곱합을 계산
- `backward`: 오차역전파 수행
  - 출력층 델타(`-2 * (target - output)`)부터 시작해 역순으로 각 층의 델타 계산
  - 델타로부터 가중치의 기울기(`dw`)와 편향의 기울기(`db`)를 구하고, 학습률만큼 가중치·편향을 갱신
- `train`: 지정한 횟수만큼 순전파 → 오차 출력 → 역전파를 반복하고, 마지막에 한 번 더 순전파를 수행해 최종 결과와 오차를 출력

### 입력
- 이전 셀에서 `sys.stdin`을 `io.StringIO`로 미리 채워둔 덕분에, `sys.stdin.read().split()`로 표준입력 전체를 공백 기준 토큰으로 한 번에 읽어 순서대로 소비
- 읽는 순서: 레이어 개수 → 각 층의 노드 수 → 편향 → 학습률 → 입력값 → 층 사이 가중치 행렬들 → 목표값 → 반복 횟수
- 각 가중치 행렬은 `(다음 층 노드 수) x (이전 층 노드 수)` 크기로 값을 읽어 `reshape` 후 리스트에 저장


In [8]:

import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(a):
    return a * (1 - a)


def fmt_vec(v):
    return "[" + ", ".join(f"{x:.4f}" for x in v.flatten()) + "]"

def fmt_mat(m):
    return "\n".join("    " + fmt_vec(row) for row in m)


class MLP:
    def __init__(self, b, lr, nodes=[3, 3, 3]):
        self.layers = [np.zeros((i, 1)) for i in nodes]
        self.bias = b
        self.learning_rate = lr

    def setting_input_layer(self, input_layer=[0.1, 0.2, 0.3]):
        self.layers[0] = np.array(input_layer).reshape(-1, 1)

    def setting_weights(self, weight):
        self.weights = weight

    def forward(self):
        for i in range(len(self.layers) - 1):
            self.layers[i + 1] = self.weights[i] @ self.layers[i]
            if i != len(self.layers) - 2:
                self.layers[i + 1] = sigmoid(self.layers[i + 1] + self.bias)
                #self.layers[i + 1] = np.round(self.layers[i + 1],2)
            else:
                self.layers[i + 1] = self.layers[i + 1] + self.bias
            

        print("[순전파 결과]")
        n = len(self.layers)
        for idx, layer in enumerate(self.layers):
            if idx == 0:
                name = "입력층"
            elif idx == n - 1:
                name = "출력층"
            else:
                name = f"은닉층{idx}"
            print(f"  {name} (layer {idx}): {fmt_vec(layer)}")
        

    def error(self, target):
        return np.sum((target - self.layers[-1]) ** 2)

    def backward(self, target):
        n = len(self.weights)
        delta = [None] * n
        dw = [None] * n

        delta[n - 1] = -2.0 * (target - self.layers[-1])
        #delta[n-1] = np.round(delta[n-1],2)

        for i in range(n - 2, -1, -1):
            delta[i] = (self.weights[i + 1].T @ delta[i + 1]) * sigmoid_deriv(self.layers[i + 1])
            #delta[i] = np.round(delta[i],2)

        db = sum(np.sum(d) for d in delta)

        for i in range(n):
            dw[i] = delta[i] @ self.layers[i].T
            #dw[i] = np.round(dw[i],2)

        self.bias -= self.learning_rate * db
        #self.bias = round(self.bias,2)
        for i in range(n):
            self.weights[i] -= self.learning_rate * dw[i]
            #self.weights[i] = np.round(weights[i],2)

        print("[역전파 결과]")
        print(" 델타:")
        for idx, d in enumerate(delta):
            print(f"  delta[{idx}]: {fmt_vec(d)}")

        print(" 기울기(dw):")
        for idx, d in enumerate(dw):
            print(f"  dw[{idx}] (shape={d.shape}):")
            print(fmt_mat(d))

        print(f" 업데이트된 편향: {self.bias:.4f}")

        print(" 업데이트된 가중치:")
        for idx, w in enumerate(self.weights):
            print(f"  weights[{idx}] (shape={w.shape}):")
            print(fmt_mat(w))

        return delta, dw, db

    def train(self, rep, target):
        for i in range(rep):
            print(f"\n{'='*10} 순전파 {i + 1} 회차 {'='*10}")
            self.forward()
            print(f"오차: {self.error(target)}")
            print(f"\n{'='*10} 역전파 {i + 1} 회차 {'='*10}")
            self.backward(target)
        print(f"\n{'='*10} 순전파 {rep + 1} 회차 {'='*10}")
        self.forward()
        print(f"오차: {self.error(target)}")


data = iter(sys.stdin.read().split())

num_layer = int(next(data))
num_nodes = [int(next(data)) for _ in range(num_layer)]
b = float(next(data))
lr = float(next(data))
m = MLP(b, lr, num_nodes)

input_layer = [float(next(data)) for _ in range(num_nodes[0])]
m.setting_input_layer(input_layer)

weights = [None] * (num_layer - 1)
for i in range(num_layer - 1):
    size = num_nodes[i + 1] * num_nodes[i]
    temp = [float(next(data)) for _ in range(size)]
    temp = np.array(temp).reshape(num_nodes[i + 1], num_nodes[i])
    weights[i] = temp
m.setting_weights(weights)

target = [float(next(data)) for _ in range(num_nodes[-1])]
target = np.array(target).reshape(-1, 1)
rep = int(next(data))

m.train(rep, target)


========== 순전파 1 회차 ==========
[순전파 결과]
  입력층 (layer 0): [0.5000, 0.3000, 0.8000]
  은닉층1 (layer 1): [0.6106, 0.7171, 0.8038]
  은닉층2 (layer 2): [0.6811, 0.8019]
  출력층 (layer 3): [0.6251, 0.9217]
오차: 0.9900572414429615

========== 역전파 1 회차 ==========
[역전파 결과]
 델타:
  delta[0]: [0.0224, 0.0248, 0.0237]
  delta[1]: [0.1513, 0.1281]
  delta[2]: [-0.7498, 1.8434]
 기울기(dw):
  dw[0] (shape=(3, 3)):
    [0.0112, 0.0067, 0.0179]
    [0.0124, 0.0074, 0.0198]
    [0.0118, 0.0071, 0.0189]
  dw[1] (shape=(2, 3)):
    [0.0924, 0.1085, 0.1216]
    [0.0782, 0.0918, 0.1029]
  dw[2] (shape=(2, 2)):
    [-0.5107, -0.6013]
    [1.2555, 1.4782]
 업데이트된 편향: -0.0444
 업데이트된 가중치:
  weights[0] (shape=(3, 3)):
    [0.0989, 0.1993, 0.2982]
    [0.3988, 0.4993, 0.5980]
    [0.6988, 0.7993, 0.8981]
  weights[1] (shape=(2, 3)):
    [0.1908, 0.2891, 0.3878]
    [0.4922, 0.5908, 0.6897]
  weights[2] (shape=(2, 2)):
    [0.3511, 0.4601]
    [0.3745, 0.4522]

========== 순전파 2 회차 ==========
[순전파 결과]
  입력층 (layer 0): [0.500